In [1]:
import numpy as np
import pandas as pd
from scipy.stats import norm

# ====================== YOUR INPUTS ======================
service_level = 0.95          # Change anytime (0.90, 0.98, 0.99...)
lead_time_days = 5
holding_cost_per_unit = 20    # ₹ per unit per month

In [2]:
# ====================== YOUR DATA FROM sku.csv ======================
data = {
    'SKU A (Low Var – T-shirts)': [
        104,98,106,115,97,97,115,107,95,105,95,95,102,80,82,
        94,89,103,90,85,114,97,100,85,94,101,88,103,93,97
    ],
    'SKU B (Med Var – Jeans)': [
        84,146,99,73,120,69,105,51,66,104,118,104,97,92,63,
        82,88,126,108,55,108,90,83,115,125,123,79,92,108,124
    ],
    'SKU C (High Var – Umbrellas)': [
        76,90,44,40,140,167,96,150,118,67,118,176,98,178,0,
        141,104,85,104,0,89,117,173,74,59,74,145,116,73,125
    ]
}

df = pd.DataFrame(data)

In [3]:
# ====================== CALCULATIONS ======================
z_score = norm.ppf(service_level)   # Exact Python version of Excel's =NORMSINV()

results = []
for sku in df.columns:
    avg_demand = df[sku].mean()
    std_dev = df[sku].std(ddof=1)   # Matches Excel STDEV.S
    
    safety_stock = z_score * std_dev * np.sqrt(lead_time_days)
    reorder_point = (avg_demand * lead_time_days) + safety_stock
    monthly_cost = safety_stock * holding_cost_per_unit
    
    results.append({
        'SKU': sku,
        'Avg Daily Demand': round(avg_demand, 1),
        'Std Dev (Variability)': round(std_dev, 1),
        'Z-Score': round(z_score, 4),
        'Safety Stock': round(safety_stock),
        'Reorder Point': round(reorder_point),
        'Monthly Holding Cost (₹)': round(monthly_cost)
    })

In [4]:
# ====================== FINAL OUTPUT ======================
result_table = pd.DataFrame(results)
print("\n" + "="*90)
print("🚀 SAFETY STOCK & REORDER POINT - YOUR sku.csv")
print("="*90)
print(result_table.to_string(index=False))
print("="*90)
print(f"✅ Service Level : {service_level*100}%")
print(f"✅ Z-Score        : {z_score:.4f}")
print(f"✅ Lead Time      : {lead_time_days} days")
print("\n💡 Insight: High-variability SKU C needs ~5× more safety stock → ₹3,453 extra monthly holding cost!")


🚀 SAFETY STOCK & REORDER POINT - YOUR sku.csv
                         SKU  Avg Daily Demand  Std Dev (Variability)  Z-Score  Safety Stock  Reorder Point  Monthly Holding Cost (₹)
  SKU A (Low Var – T-shirts)              97.5                    9.1   1.6449            33            521                       667
     SKU B (Med Var – Jeans)              96.6                   23.3   1.6449            86            568                      1712
SKU C (High Var – Umbrellas)             101.2                   46.9   1.6449           173            679                      3453
✅ Service Level : 95.0%
✅ Z-Score        : 1.6449
✅ Lead Time      : 5 days

💡 Insight: High-variability SKU C needs ~5× more safety stock → ₹3,453 extra monthly holding cost!
